# SARIMAX — Short-Term Load Forecasting with Hijri-Calendar Ablation

**Project**: Zero-shot TSFM Benchmark under Hijri-Calendar Regime Shifts  
**Model**: SARIMAX — `SARIMA(p,d,q)(P,D,Q)_24` with exogenous regressors  
**Order Selection**: AICc stepwise search  
**Ablation A**: ± Hijri features (`is_ramadan`, `is_eid`, `day_of_ramadan`)  

---
### Notebook Structure
1. Imports & Config  
2. Data Loading & Splits  
3. Exploratory Data Analysis  
4. Feature Engineering  
5. AICc-based Order Selection  
6. SARIMAX Training (Baseline — no Hijri)  
7. SARIMAX Training (Ablation — with Hijri)  
8. 24-hour Ahead Forecasting  
9. Regime-Conditional Evaluation (MAE, MAPE, RMSE, MASE)  
10. Diebold-Mariano Test  
11. Results Visualisation  
12. Save Results

## 1. Imports & Config

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import warnings
import itertools
from pathlib import Path

# ── Data ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Modelling ─────────────────────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH   = Path('1778889600899_epias_processed_final.csv')   # update if needed
OUTPUT_DIR  = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Global config ─────────────────────────────────────────────────────────────
SEASONAL_PERIOD = 24          # daily seasonality (hourly data)
FORECAST_HORIZON = 24         # 24-h ahead
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('All imports successful.')

## 2. Data Loading & Train / Val / Test Splits

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# Make timestamp timezone-naive for cleaner indexing
df['timestamp'] = df['timestamp'].dt.tz_localize(None)
df = df.set_index('timestamp')
df.index.freq = 'h'          # assert hourly frequency

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df.index.min()}  →  {df.index.max()}')
print(f'Ramadan hours : {df.is_ramadan.sum():,}  ({df.is_ramadan.mean()*100:.1f}%)')
df.head(3)

In [ ]:
# ── Train / Val / Test splits (matching Section 3 of the paper) ───────────────
train = df.loc[:'2022-12-31']
val   = df.loc['2023-01-01':'2023-12-31']
test  = df.loc['2024-01-01':]

print(f'Train : {train.index.min().date()}  →  {train.index.max().date()}  ({len(train):,} obs)')
print(f'Val   : {val.index.min().date()}  →  {val.index.max().date()}  ({len(val):,} obs)')
print(f'Test  : {test.index.min().date()}  →  {test.index.max().date()}  ({len(test):,} obs)')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# ── Panel 1: Full load series ─────────────────────────────────────────────────
axes[0].plot(df.index, df['actual_load'] / 1e3, color='steelblue', linewidth=0.4)
ramadan_mask = df['is_ramadan'].astype(bool)
axes[0].fill_between(df.index, 0, df['actual_load'].max() / 1e3,
                     where=ramadan_mask, alpha=0.2, color='orange', label='Ramadan')
axes[0].set_ylabel('Load (GW)')
axes[0].set_title('Hourly National Load — Full Series (EPIAS)')
axes[0].legend()

# ── Panel 2: Average load by hour — Normal vs Ramadan ─────────────────────────
hourly_normal  = df[df.is_ramadan == 0].groupby('hour')['actual_load'].mean()
hourly_ramadan = df[df.is_ramadan == 1].groupby('hour')['actual_load'].mean()
axes[1].plot(hourly_normal.index,  hourly_normal  / 1e3, label='Normal',  marker='o', ms=4)
axes[1].plot(hourly_ramadan.index, hourly_ramadan / 1e3, label='Ramadan', marker='s', ms=4, color='orange')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Load (GW)')
axes[1].set_title('Average Diurnal Profile — Normal vs. Ramadan')
axes[1].legend()
axes[1].set_xticks(range(24))

# ── Panel 3: ACF on training series (first 168 lags = 1 week) ─────────────────
plot_acf(train['actual_load'], lags=168, ax=axes[2], alpha=0.05)
axes[2].set_title('ACF — Training Series (168 lags = 1 week)')
axes[2].set_xlabel('Lag (hours)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', bbox_inches='tight')
plt.show()
print('Figure saved to outputs/eda_overview.png')

In [ ]:
# ── Stationarity tests ────────────────────────────────────────────────────────
# Use a daily-sampled series (midnight obs) to keep ADF fast
daily = train['actual_load'].iloc[::24]

adf_stat, adf_p, *_ = adfuller(daily, autolag='AIC')
print(f'ADF  — stat: {adf_stat:.3f},  p-value: {adf_p:.4f}  →  '
      f'{"Stationary" if adf_p < 0.05 else "Non-stationary"}')

kpss_stat, kpss_p, *_ = kpss(daily, regression='c', nlags='auto')
print(f'KPSS — stat: {kpss_stat:.3f},  p-value: {kpss_p:.4f}  →  '
      f'{"Stationary" if kpss_p > 0.05 else "Non-stationary"}')

## 4. Feature Engineering

We define two exogenous matrices for **Ablation A**:

| Config | Columns |
|---|---|
| **Baseline** (no Hijri) | `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos` |
| **+ Hijri** | baseline + `is_ramadan`, `is_eid`, `day_of_ramadan_norm` |

In [ ]:
def cyclical_encode(series: pd.Series, period: int) -> pd.DataFrame:
    """Encode a periodic feature as sin/cos pair."""
    name = series.name
    return pd.DataFrame({
        f'{name}_sin': np.sin(2 * np.pi * series / period),
        f'{name}_cos': np.cos(2 * np.pi * series / period),
    }, index=series.index)


def build_exog(frame: pd.DataFrame, include_hijri: bool = False) -> pd.DataFrame:
    """Build exogenous regressor matrix from a data frame.

    Parameters
    ----------
    frame        : slice of the master dataframe
    include_hijri: whether to append Hijri calendar features
    """
    parts = [
        cyclical_encode(frame['hour'],       period=24),
        cyclical_encode(frame['day_of_week'], period=7),
        cyclical_encode(frame['month'],       period=12),
    ]

    if include_hijri:
        hijri = pd.DataFrame({
            'is_ramadan'        : frame['is_ramadan'].astype(float),
            'is_eid'            : frame['is_eid'].astype(float),
            # Normalise day-of-Ramadan to [0,1]; 0 outside Ramadan
            'day_of_ramadan_norm': frame['day_of_ramadan'] / 30.0,
        }, index=frame.index)
        parts.append(hijri)

    return pd.concat(parts, axis=1)


# ── Build matrices for every split × every config ─────────────────────────────
exog_configs = {
    'baseline': False,   # no Hijri
    'hijri'   : True,    # with Hijri
}

for split_name, split in [('train', train), ('val', val), ('test', test)]:
    for cfg_name, inc_hijri in exog_configs.items():
        mat = build_exog(split, include_hijri=inc_hijri)
        print(f'{split_name:5s} | {cfg_name:8s} | shape: {mat.shape} | cols: {mat.columns.tolist()}')

## 5. AICc-Based Order Selection

We use a stepwise grid search over a coarse SARIMA order space on the **training series** (daily-sampled at midnight for speed), then confirm on the hourly series with the best `(p,d,q)` and seasonal orders `(P,D,Q)_24 = (1,1,1)` as is standard for hourly load.

In [ ]:
def aicc(model_result) -> float:
    """AICc = AIC + 2k(k+1)/(n-k-1) where k = num params, n = obs."""
    k = model_result.df_model + 1          # +1 for sigma^2
    n = model_result.nobs
    return model_result.aic + (2 * k * (k + 1)) / max(n - k - 1, 1)


def stepwise_order_search(
    endog: pd.Series,
    exog: pd.DataFrame,
    p_range=(0, 3),
    q_range=(0, 3),
    d: int = 1,
    seasonal_order: tuple = (1, 1, 1, 24),
) -> dict:
    """
    Stepwise AICc search over (p, q) space with fixed d and seasonal order.
    Returns best order and the results table.
    """
    results = []
    ps = range(*p_range)
    qs = range(*q_range)
    total = len(ps) * len(qs)
    print(f'Searching {total} (p,q) combinations ...')

    for p, q in itertools.product(ps, qs):
        try:
            m = SARIMAX(
                endog,
                exog=exog,
                order=(p, d, q),
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False, maxiter=200)
            results.append({'p': p, 'd': d, 'q': q, 'aicc': aicc(m), 'aic': m.aic, 'bic': m.bic})
        except Exception as e:
            results.append({'p': p, 'd': d, 'q': q, 'aicc': np.inf, 'aic': np.inf, 'bic': np.inf})

    df_res = pd.DataFrame(results).sort_values('aicc').reset_index(drop=True)
    best   = df_res.iloc[0]
    print(f'\nBest order → p={int(best.p)}, d={int(best.d)}, q={int(best.q)}  |  AICc={best.aicc:.2f}')
    return {'order': (int(best.p), d, int(best.q)), 'table': df_res}


print('Running AICc order search on DAILY-sampled training data (fast proxy) ...')
# Daily-sampled (midnight obs) for speed; orders transfer to hourly model
train_daily      = train['actual_load'].iloc[::24]
exog_daily_base  = build_exog(train.iloc[::24], include_hijri=False)

search_result    = stepwise_order_search(
    endog          = train_daily,
    exog           = exog_daily_base,
    p_range        = (0, 4),
    q_range        = (0, 4),
    d              = 1,
    seasonal_order = (1, 1, 1, 7),   # 7-day weekly on daily series
)

BEST_P, BEST_D, BEST_Q = search_result['order']
search_result['table'].head(10)

In [ ]:
# ── Visualise AICc surface ─────────────────────────────────────────────────────
tbl = search_result['table'].copy()
tbl = tbl[tbl['aicc'] < np.inf]

pivot = tbl.pivot(index='p', columns='q', values='aicc')

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax, linewidths=0.4)
ax.set_title(f'AICc by (p, q)  |  d={BEST_D}  |  Best: p={BEST_P}, q={BEST_Q}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'aicc_surface.png', bbox_inches='tight')
plt.show()

## 6. SARIMAX Training — Baseline (no Hijri features)

In [ ]:
# ── Fit on TRAIN, use SEASONAL order (P,D,Q)_24 = (1,1,1) for hourly ─────────
SEASONAL_ORDER = (1, 1, 1, SEASONAL_PERIOD)

print(f'Fitting SARIMAX({BEST_P},{BEST_D},{BEST_Q})x{SEASONAL_ORDER}  [BASELINE — no Hijri] ...')

model_base = SARIMAX(
    train['actual_load'],
    exog           = build_exog(train, include_hijri=False),
    order          = (BEST_P, BEST_D, BEST_Q),
    seasonal_order = SEASONAL_ORDER,
    enforce_stationarity = False,
    enforce_invertibility= False,
)

fit_base = model_base.fit(disp=False, maxiter=500)
print(fit_base.summary())

In [ ]:
# ── Residual diagnostics ──────────────────────────────────────────────────────
fig = fit_base.plot_diagnostics(figsize=(14, 8))
fig.suptitle('SARIMAX Baseline — Residual Diagnostics', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residuals_baseline.png', bbox_inches='tight')
plt.show()

# Ljung-Box test on residuals (lags 24 and 48)
lb = acorr_ljungbox(fit_base.resid.dropna(), lags=[24, 48], return_df=True)
print('Ljung-Box test (lags 24, 48):')
print(lb)

## 7. SARIMAX Training — Ablation (with Hijri features)

In [ ]:
print(f'Fitting SARIMAX({BEST_P},{BEST_D},{BEST_Q})x{SEASONAL_ORDER}  [HIJRI — with Ramadan features] ...')

model_hijri = SARIMAX(
    train['actual_load'],
    exog           = build_exog(train, include_hijri=True),
    order          = (BEST_P, BEST_D, BEST_Q),
    seasonal_order = SEASONAL_ORDER,
    enforce_stationarity = False,
    enforce_invertibility= False,
)

fit_hijri = model_hijri.fit(disp=False, maxiter=500)
print(fit_hijri.summary())

In [ ]:
# ── Compare Hijri coefficient significance ────────────────────────────────────
hijri_cols = ['is_ramadan', 'is_eid', 'day_of_ramadan_norm']
coef_df = pd.DataFrame({
    'coef'    : fit_hijri.params[hijri_cols],
    'std_err' : fit_hijri.bse[hijri_cols],
    'z'       : fit_hijri.tvalues[hijri_cols],
    'p_value' : fit_hijri.pvalues[hijri_cols],
})
print('\nHijri regressor coefficients (Ablation A):')
print(coef_df.to_string())

## 8. 24-Hour Ahead Forecasting (Rolling / One-Shot)

We use **one-step-ahead rolling forecasts** over the test set:  
at each hour `t`, we produce a 24-step forecast using all history up to `t`, then advance by 24 hours.  
This mirrors real operational scheduling (unit commitment, next-day dispatch).

In [ ]:
def rolling_forecast_24h(
    fit_result,
    test_frame: pd.DataFrame,
    include_hijri: bool,
    horizon: int = 24,
) -> pd.Series:
    """
    Produce 24-step-ahead rolling forecasts over the test set.
    At each step we advance by `horizon` hours (non-overlapping windows).

    Returns a pd.Series of forecasts aligned to the test index.
    """
    preds = []
    n_windows = len(test_frame) // horizon

    model_upd = fit_result          # updated model handle

    for i in range(n_windows):
        window_start = i * horizon
        window_end   = window_start + horizon
        forecast_idx = test_frame.index[window_start:window_end]

        exog_fc = build_exog(test_frame.iloc[window_start:window_end],
                             include_hijri=include_hijri)

        fc = model_upd.forecast(steps=horizon, exog=exog_fc)
        preds.append(fc)

        # Append actuals so the state carries forward
        actual_window   = test_frame['actual_load'].iloc[window_start:window_end]
        exog_actual     = build_exog(test_frame.iloc[window_start:window_end],
                                     include_hijri=include_hijri)
        model_upd = model_upd.extend(actual_window, exog=exog_actual)

    forecast_series = pd.concat(preds)
    return forecast_series


print('Generating rolling 24-h forecasts on TEST set ...')
print('  → Baseline (no Hijri) ...')
fcst_base  = rolling_forecast_24h(fit_base,  test, include_hijri=False)

print('  → Ablation (+ Hijri) ...')
fcst_hijri = rolling_forecast_24h(fit_hijri, test, include_hijri=True)

print(f'Forecast length: {len(fcst_base):,} hours')

In [ ]:
# ── Align forecasts with test actuals ─────────────────────────────────────────
# Trim test to the forecasted portion (last partial window dropped)
n_fc     = len(fcst_base)
test_aln = test.iloc[:n_fc].copy()

test_aln['fcst_base']  = fcst_base.values
test_aln['fcst_hijri'] = fcst_hijri.values

print('Aligned test frame shape:', test_aln.shape)
test_aln[['actual_load', 'fcst_base', 'fcst_hijri']].head()

## 9. Regime-Conditional Evaluation

Four regimes (Table 3 of the paper):

| Regime | Definition |
|---|---|
| **Normal** | Non-Ramadan, no heatwave |
| **Ramadan** | `is_ramadan == 1` |
| **Heatwave** | *(requires T_max — skipped if column absent)* |
| **Compound** | Ramadan AND heatwave |

In [ ]:
def seasonal_naive_lag168(frame: pd.DataFrame) -> pd.Series:
    """Lag-168 seasonal naive forecast (same hour last week)."""
    return frame['load_lag_168h']


def mase(actual: np.ndarray, forecast: np.ndarray, naive: np.ndarray) -> float:
    """Mean Absolute Scaled Error vs. seasonal naive."""
    mae_model = np.mean(np.abs(actual - forecast))
    mae_naive = np.mean(np.abs(actual - naive))
    return mae_model / (mae_naive + 1e-9)


def mape(actual: np.ndarray, forecast: np.ndarray) -> float:
    return np.mean(np.abs((actual - forecast) / (actual + 1e-9))) * 100


def evaluate(frame: pd.DataFrame, pred_col: str, label: str) -> dict:
    """Compute MAE, MAPE, RMSE, MASE for a given prediction column."""
    y     = frame['actual_load'].values
    yhat  = frame[pred_col].values
    naive = seasonal_naive_lag168(frame).values
    return {
        'Config' : label,
        'N'      : len(y),
        'MAE'    : mean_absolute_error(y, yhat),
        'MAPE'   : mape(y, yhat),
        'RMSE'   : np.sqrt(mean_squared_error(y, yhat)),
        'MASE'   : mase(y, yhat, naive),
    }


# ── Define regimes ─────────────────────────────────────────────────────────────
regimes = {
    'Normal'  : test_aln[test_aln['is_ramadan'] == 0],
    'Ramadan' : test_aln[test_aln['is_ramadan'] == 1],
}

# ── Collect metrics ────────────────────────────────────────────────────────────
rows = []
for regime_name, regime_df in regimes.items():
    if len(regime_df) == 0:
        continue
    rows.append({'Regime': regime_name, **evaluate(regime_df, 'fcst_base',  'Baseline (no Hijri)')})
    rows.append({'Regime': regime_name, **evaluate(regime_df, 'fcst_hijri', 'SARIMAX + Hijri')})

# Aggregate (all test)
rows.append({'Regime': 'All Test', **evaluate(test_aln, 'fcst_base',  'Baseline (no Hijri)')})
rows.append({'Regime': 'All Test', **evaluate(test_aln, 'fcst_hijri', 'SARIMAX + Hijri')})

metrics_df = pd.DataFrame(rows).set_index(['Regime', 'Config'])
metrics_df = metrics_df.round({'MAE': 1, 'MAPE': 3, 'RMSE': 1, 'MASE': 4})

print('\n=== Regime-Conditional Evaluation Results ===')
print(metrics_df.to_string())

In [ ]:
# ── Delta table: how much do Hijri features help? ─────────────────────────────
delta_rows = []
for regime_name, regime_df in {**regimes, 'All Test': test_aln}.items():
    if len(regime_df) == 0:
        continue
    m_base  = evaluate(regime_df, 'fcst_base',  'Baseline')
    m_hijri = evaluate(regime_df, 'fcst_hijri', 'Hijri')
    delta_rows.append({
        'Regime'    : regime_name,
        'ΔMAE (MW)' : m_hijri['MAE']  - m_base['MAE'],
        'ΔMAPE (%)'  : m_hijri['MAPE'] - m_base['MAPE'],
        'ΔRMSE (MW)': m_hijri['RMSE'] - m_base['RMSE'],
        'ΔMASE'     : m_hijri['MASE'] - m_base['MASE'],
    })

delta_df = pd.DataFrame(delta_rows).set_index('Regime').round(3)
print('\n=== Hijri Feature Marginal Value  (negative = improvement) ===')
print(delta_df.to_string())

## 10. Diebold-Mariano Test

One-sided DM test: H₀ — equal predictive accuracy; H₁ — Hijri model is **more accurate** (smaller loss).  
Newey-West HAC standard errors; Holm-Bonferroni correction across regimes.

In [ ]:
from scipy import stats


def diebold_mariano(
    e1: np.ndarray,
    e2: np.ndarray,
    h: int = 24,
    power: int = 2,
) -> dict:
    """
    Diebold-Mariano test with Newey-West HAC standard errors.

    Parameters
    ----------
    e1, e2 : forecast errors of model 1 (baseline) and model 2 (Hijri)
    h      : forecast horizon for bandwidth selection
    power  : loss differential power (1 = MAE, 2 = MSE)

    Returns
    -------
    dict with DM statistic, p-value, and conclusion
    """
    d  = np.abs(e1) ** power - np.abs(e2) ** power   # loss differential
    T  = len(d)
    d_bar = d.mean()

    # Newey-West HAC variance (bandwidth = h - 1)
    bw   = max(1, h - 1)
    gamma_0 = np.mean((d - d_bar) ** 2)
    gamma_sum = sum(
        (1 - lag / (bw + 1)) * np.mean((d[lag:] - d_bar) * (d[:-lag] - d_bar))
        for lag in range(1, bw + 1)
        if lag < T
    )
    var_d = (gamma_0 + 2 * gamma_sum) / T
    var_d = max(var_d, 1e-10)    # numerical guard

    dm_stat = d_bar / np.sqrt(var_d)
    # One-sided: H1 = model 2 (Hijri) more accurate → d > 0 → right tail
    p_value = 1 - stats.norm.cdf(dm_stat)

    return {
        'DM stat' : round(dm_stat, 4),
        'p-value' : round(p_value, 4),
        'Reject H0 (α=0.05)': p_value < 0.05,
    }


# ── Run DM tests per regime ────────────────────────────────────────────────────
dm_results = []
for regime_name, regime_df in {**regimes, 'All Test': test_aln}.items():
    if len(regime_df) < 50:
        continue
    e_base  = (regime_df['actual_load'] - regime_df['fcst_base']).values
    e_hijri = (regime_df['actual_load'] - regime_df['fcst_hijri']).values
    dm      = diebold_mariano(e_base, e_hijri, h=FORECAST_HORIZON, power=1)
    dm_results.append({'Regime': regime_name, **dm})

dm_df = pd.DataFrame(dm_results).set_index('Regime')

# Holm-Bonferroni correction
p_vals = dm_df['p-value'].values
n_tests = len(p_vals)
sorted_idx = np.argsort(p_vals)
hb_reject  = np.zeros(n_tests, dtype=bool)
for rank, idx in enumerate(sorted_idx):
    if p_vals[idx] <= 0.05 / (n_tests - rank):
        hb_reject[idx] = True
    else:
        break
dm_df['HB Reject (α=0.05)'] = hb_reject

print('\n=== Diebold-Mariano Test Results (H1: Hijri model more accurate) ===')
print(dm_df.to_string())

## 11. Results Visualisation

In [ ]:
# ── Fig 1: Forecast vs Actual — full test period ──────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(test_aln.index, test_aln['actual_load'] / 1e3,
        label='Actual', color='black', linewidth=0.5, alpha=0.8)
ax.plot(test_aln.index, test_aln['fcst_base'] / 1e3,
        label='Baseline (no Hijri)', color='steelblue', linewidth=0.5, alpha=0.7)
ax.plot(test_aln.index, test_aln['fcst_hijri'] / 1e3,
        label='SARIMAX + Hijri', color='darkorange', linewidth=0.5, alpha=0.7)

# Shade Ramadan windows
for _, g in test_aln[test_aln['is_ramadan'] == 1].groupby(
        (test_aln['is_ramadan'] != test_aln['is_ramadan'].shift()).cumsum()):
    ax.axvspan(g.index[0], g.index[-1], alpha=0.12, color='orange')

ax.set_ylabel('Load (GW)')
ax.set_title('SARIMAX 24-h Ahead Forecasts — Test Set (2024–2025)\nOrange shading = Ramadan')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'forecast_test_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Zoom into a 2-week Ramadan window ──────────────────────────────────
ram_rows   = test_aln[test_aln['is_ramadan'] == 1]
if len(ram_rows) >= 336:
    zoom_start = ram_rows.index[0]
    zoom_end   = zoom_start + pd.Timedelta(hours=336)   # 14 days
    zoom       = test_aln.loc[zoom_start:zoom_end]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    for ax, (pred_col, label, color) in zip(axes, [
        ('fcst_base',  'Baseline (no Hijri)', 'steelblue'),
        ('fcst_hijri', 'SARIMAX + Hijri',     'darkorange'),
    ]):
        ax.plot(zoom.index, zoom['actual_load'] / 1e3, color='black',
                label='Actual', linewidth=1.2)
        ax.plot(zoom.index, zoom[pred_col]    / 1e3, color=color,
                label=label, linewidth=1.2, linestyle='--')
        err = (zoom['actual_load'] - zoom[pred_col]).abs()
        ax.fill_between(zoom.index,
                        zoom['actual_load'] / 1e3,
                        zoom[pred_col]     / 1e3,
                        alpha=0.2, color=color)
        ax.set_ylabel('Load (GW)')
        ax.set_title(f'{label}  |  MAE: {err.mean():.0f} MW')
        ax.legend()

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    fig.suptitle('2-Week Ramadan Zoom — SARIMAX Baseline vs. Hijri Model', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ramadan_zoom.png', bbox_inches='tight')
    plt.show()
else:
    print('Not enough Ramadan test data for zoom plot.')

In [ ]:
# ── Fig 3: Regime-conditional MAE bar chart ────────────────────────────────────
plot_df = metrics_df.reset_index()
fig, ax = plt.subplots(figsize=(9, 5))

regimes_plot = plot_df['Regime'].unique()
x = np.arange(len(regimes_plot))
width = 0.35

base_mae  = plot_df[plot_df['Config'] == 'Baseline (no Hijri)'].set_index('Regime')['MAE']
hijri_mae = plot_df[plot_df['Config'] == 'SARIMAX + Hijri'].set_index('Regime')['MAE']

bars1 = ax.bar(x - width/2, [base_mae[r]  for r in regimes_plot],
               width, label='Baseline (no Hijri)', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, [hijri_mae[r] for r in regimes_plot],
               width, label='SARIMAX + Hijri',     color='darkorange', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(regimes_plot)
ax.set_ylabel('MAE (MW)')
ax.set_title('Regime-Conditional MAE — Ablation A (± Hijri Features)')
ax.legend()
ax.bar_label(bars1, fmt='%.0f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.0f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regime_mae_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Error distribution — Ramadan only ──────────────────────────────────
ram_test = test_aln[test_aln['is_ramadan'] == 1].copy()
if len(ram_test) > 0:
    ram_test['err_base']  = ram_test['actual_load'] - ram_test['fcst_base']
    ram_test['err_hijri'] = ram_test['actual_load'] - ram_test['fcst_hijri']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(ram_test['err_base']  / 1e3, bins=50, alpha=0.6, label='Baseline',      color='steelblue')
    ax.hist(ram_test['err_hijri'] / 1e3, bins=50, alpha=0.6, label='+ Hijri',       color='darkorange')
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel('Forecast Error (GW)')
    ax.set_ylabel('Count')
    ax.set_title('Forecast Error Distribution — Ramadan Period Only')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'error_distribution_ramadan.png', bbox_inches='tight')
    plt.show()
else:
    print('No Ramadan observations in test set.')

In [ ]:
# ── Fig 5: Error by hour of day — Normal vs Ramadan ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (regime_name, regime_df) in zip(axes, regimes.items()):
    if len(regime_df) == 0:
        ax.set_visible(False)
        continue
    hourly_err_base  = regime_df.groupby('hour').apply(
        lambda g: mean_absolute_error(g['actual_load'], g['fcst_base']))
    hourly_err_hijri = regime_df.groupby('hour').apply(
        lambda g: mean_absolute_error(g['actual_load'], g['fcst_hijri']))

    ax.plot(hourly_err_base.index,  hourly_err_base  / 1e3, label='Baseline', marker='o', ms=4)
    ax.plot(hourly_err_hijri.index, hourly_err_hijri / 1e3, label='+ Hijri',  marker='s', ms=4, color='darkorange')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('MAE (GW)')
    ax.set_title(f'{regime_name} — Hourly MAE Profile')
    ax.set_xticks(range(24))
    ax.legend()

plt.suptitle('Hourly MAE Profile by Regime — SARIMAX Baseline vs. + Hijri', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hourly_mae_profile.png', bbox_inches='tight')
plt.show()

## 12. Save Results

In [ ]:
# ── Save metrics tables ────────────────────────────────────────────────────────
metrics_df.to_csv(OUTPUT_DIR / 'sarimax_metrics.csv')
delta_df.to_csv(OUTPUT_DIR  / 'sarimax_delta_metrics.csv')
dm_df.to_csv(OUTPUT_DIR     / 'sarimax_dm_test.csv')

# ── Save forecast series ───────────────────────────────────────────────────────
test_aln[['actual_load', 'fcst_base', 'fcst_hijri', 'is_ramadan', 'is_eid',
           'day_of_ramadan', 'hour', 'day_of_week', 'month']].to_csv(
    OUTPUT_DIR / 'sarimax_forecasts.csv'
)

# ── Summary print ──────────────────────────────────────────────────────────────
print('=== SARIMAX SUMMARY ===')
print(f'Model order  : SARIMA({BEST_P},{BEST_D},{BEST_Q}) × (1,1,1)_24')
print(f'Exog (base)  : hour/dow/month cyclical sin-cos')
print(f'Exog (hijri) : + is_ramadan, is_eid, day_of_ramadan_norm')
print()
print(metrics_df.to_string())
print()
print('Diebold-Mariano (H1: Hijri more accurate):')
print(dm_df.to_string())
print()
print(f'All outputs saved to: {OUTPUT_DIR.resolve()}')